In [0]:
%run ../utils/utils

## Feature Engineering — União (squad1 + squad3) + Tratamento de Colunas

In [0]:
import pyspark.sql.functions as F
import pandas as pd

## União squad1 + squad3

In [0]:
def ler_delta_squad3(camada: str, tabela: str, storage_opts: dict):
    caminho_final = get_delta_path_squad3(camada, tabela, storage_opts)
    account_name = storage_opts.get("account_name")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")
 
    df = (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .load(caminho_final))
 
    try:
        df.take(1)  # força a validação AGORA, dentro do try, não depois
    except Exception as e:
        raise Exception(
            f"Erro ao ler squad3/{camada}/{tabela} com OAuth explícito. "
            f"Verifique se o Service Principal tem permissão de leitura no "
            f"container 'squad3'. Detalhe original: {e}"
        )
 
    return df

In [0]:
# ------------------------------------------------------------------
# 1.1 ecommerce_pedidos
# ------------------------------------------------------------------
# NOTA: status_pedido é trazido só para uso INTERPRETATIVO posterior (ex:
# cruzar anomalias com pedidos cancelados nos insights) — NÃO entra como
# feature do modelo, para não vazar um resultado pós-fato para a rede.
colunas_pedidos_modelo = ["id_pedido", "id_cliente", "dt_pedido", "valor_total", "valor_frete", "metodo_pagamento", "status_pedido"]
 
df_pedidos_squad1 = (
    ler_delta("silver", "ecommerce_pedidos", STORAGE_OPTIONS)
    .select(*colunas_pedidos_modelo)
    .withColumn("id_pedido", F.concat(F.lit("s1_"), F.col("id_pedido").cast("string")))
    .withColumn("id_cliente", F.concat(F.lit("s1_"), F.col("id_cliente").cast("string")))
    .withColumn("origem_squad", F.lit("squad1"))
)
 
df_pedidos_squad3 = (
    ler_delta_squad3("silver", "ecommerce_pedidos", STORAGE_OPTIONS)
    .select(*colunas_pedidos_modelo)
    .withColumn("id_pedido", F.concat(F.lit("s3_"), F.col("id_pedido").cast("string")))
    .withColumn("id_cliente", F.concat(F.lit("s3_"), F.col("id_cliente").cast("string")))
    .withColumn("origem_squad", F.lit("squad3"))
)
 
df_pedidos_unificado = df_pedidos_squad1.unionByName(df_pedidos_squad3)
print(f"Pedidos squad1: {df_pedidos_squad1.count()} | squad3: {df_pedidos_squad3.count()} | unificados: {df_pedidos_unificado.count()}")

In [0]:
# ------------------------------------------------------------------
# 1.2 ecommerce_itens_pedido
# ------------------------------------------------------------------
colunas_itens_modelo = ["id_item_pedido", "id_pedido", "sku", "quantidade", "desconto_aplicado"]
 
df_itens_squad1 = (
    ler_delta("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    .select(*colunas_itens_modelo)
    .withColumn("id_item_pedido", F.concat(F.lit("s1_"), F.col("id_item_pedido").cast("string")))
    .withColumn("id_pedido", F.concat(F.lit("s1_"), F.col("id_pedido").cast("string")))
    .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku")))
)
 
df_itens_squad3 = (
    ler_delta_squad3("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    .select(*colunas_itens_modelo)
    .withColumn("id_item_pedido", F.concat(F.lit("s3_"), F.col("id_item_pedido").cast("string")))
    .withColumn("id_pedido", F.concat(F.lit("s3_"), F.col("id_pedido").cast("string")))
    .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku")))
)
 
df_itens_unificado = df_itens_squad1.unionByName(df_itens_squad3)
print(f"Itens squad1: {df_itens_squad1.count()} | squad3: {df_itens_squad3.count()} | unificados: {df_itens_unificado.count()}")

In [0]:
# ------------------------------------------------------------------
# 1.3 ecommerce_produtos
# ------------------------------------------------------------------
colunas_produtos_modelo = ["sku", "id_categoria"]
 
df_produtos_squad1 = (
    ler_delta("silver", "ecommerce_produtos", STORAGE_OPTIONS)
    .select(*colunas_produtos_modelo)
    .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku")))
    .withColumn("id_categoria", F.concat(F.lit("s1_"), F.col("id_categoria").cast("string")))
)
 
df_produtos_squad3 = (
    ler_delta_squad3("silver", "ecommerce_produtos", STORAGE_OPTIONS)
    .select(*colunas_produtos_modelo)
    .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku")))
    .withColumn("id_categoria", F.concat(F.lit("s3_"), F.col("id_categoria").cast("string")))
)
 
df_produtos_unificado = df_produtos_squad1.unionByName(df_produtos_squad3)
print(f"Produtos squad1: {df_produtos_squad1.count()} | squad3: {df_produtos_squad3.count()} | unificados: {df_produtos_unificado.count()}")

In [0]:
# 1.4 União concluída — mantida em memória (df_pedidos_unificado,
# df_itens_unificado, df_produtos_unificado), sem gravar em disco. Só a
# tabela FINAL (stg_features_engineered, na Parte 2) é persistida — evita
# ter duas tabelas separadas para a mesma responsabilidade.
# ------------------------------------------------------------------
print("União squad1+squad3 concluída em memória (não gravada em disco — só o resultado final da Parte 2 será salvo).")

##  Seleção e tratamento de colunas (Feature Engineering)

In [0]:
#  **Features derivadas** calculadas em `construir_features_pedidos()`
#  (`utils.py`): `ticket_medio_historico_cliente`, `desvio_pct_vs_media_cliente`,
#  `dias_desde_ultimo_pedido`, `qtd_pedidos_anteriores_cliente`, `hora_do_dia`,
#  `dia_da_semana`, `razao_frete_valor`, `qtd_categorias_distintas`,
#  `desconto_total`.
# 
#  **Encoding**: `metodo_pagamento` (categórica, 3 valores) vira One-Hot
#  Encoding (`pgto_pix`, `pgto_boleto`, `pgto_cartão`) — evita ordem falsa
#  entre métodos de pagamento.
# 
#  **O que NÃO é feito aqui**: a normalização/escala (`StandardScaler`) fica
#  para o notebook `modelo.py`, de propósito — ela precisa ser calculada
#  (`fit`) só em cima do conjunto de TREINO, e isso só é decidido depois do
#  split temporal, que também é responsabilidade do `modelo.py`.

df_features = construir_features_pedidos(
    df_pedidos_unificado,
    df_itens_unificado,
    df_produtos_unificado
)

In [0]:
# Traz de volta dt_pedido, origem_squad e status_pedido (não fazem parte do
# retorno de construir_features_pedidos, mas são necessários: os dois
# primeiros para o split temporal no modelo.py, o terceiro só para os
# insights posteriores — nunca usado como feature).

df_features_completo = (
    df_pedidos_unificado
    .select("id_pedido", "dt_pedido", "origem_squad", "status_pedido")
    .join(df_features, "id_pedido")
)
 
print(f"Total de pedidos com features construídas: {df_features_completo.count()}")
display(df_features_completo.limit(10))
 
 

## Encoding categórico (One-Hot) — determinístico, não depende de treino/teste

In [0]:
pdf_features = df_features_completo.toPandas()
 
# Normaliza para evitar categorias duplicadas por causa de casing
pdf_features["metodo_pagamento"] = (
    pdf_features["metodo_pagamento"]
    .astype(str)
    .str.strip()
    .str.lower()
)
 
pdf_features_encoded = pd.get_dummies(pdf_features, columns=["metodo_pagamento"], prefix="pgto")
 
 
def consolidar_colunas_duplicadas_case_insensitive(df):
    """
    Segurança extra: se por qualquer outro motivo sobrarem colunas one-hot
    que só diferem por caixa (ex: 'pgto_pix' e 'pgto_Pix'), consolida em uma
    única coluna antes de gravar — evita quebrar a leitura no Spark, que é
    case-insensitive para nomes de coluna.
    """
    grupos = {}
    for col in df.columns:
        grupos.setdefault(col.lower(), []).append(col)
 
    for nome_lower, cols in grupos.items():
        if len(cols) > 1:
            df[cols[0]] = df[cols].sum(axis=1).clip(upper=1)
            for c in cols[1:]:
                df.drop(columns=c, inplace=True)
            df.rename(columns={cols[0]: nome_lower}, inplace=True)
    return df
 
 
pdf_features_encoded = consolidar_colunas_duplicadas_case_insensitive(pdf_features_encoded)
 
colunas_onehot = sorted([c for c in pdf_features_encoded.columns if c.startswith("pgto_")])
print(f"Colunas one-hot geradas: {colunas_onehot}")

##  Salvar o resultado — entrada oficial do notebook `modelo.py`

In [0]:
df_features_final = spark.createDataFrame(pdf_features_encoded)
 
gravar_delta(
    df=df_features_final,
    camada="IA/encouders",
    tabela="stg_features_engineered",
    storage_opts=STORAGE_OPTIONS,
    mode="overwrite",
    particionar=False
)
 
print(f"Features salvas em IA/encouders/stg_features_engineered: {df_features_final.count()} pedidos, "
      f"{len(pdf_features_encoded.columns)} colunas.")
print("O notebook 'modelo.py' deve usar esta tabela como entrada — não a união bruta.")